In [1]:
!pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 54.7 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [langchain_experimental]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


# **Built-in Tools**


### **DuckDuckGOSearchRun**


In [3]:
!pip install -U ddgs

In [8]:
from langchain_community.tools import DuckDuckGoSearchRun
search_tool = DuckDuckGoSearchRun()
results = search_tool.invoke("What is the capital of Dholakpur (chotta bheem)?")
print(results)

Chhota Bheem (transl. Little Bheem) is an Indian animated comedy adventure television series, created by Green Gold Animations based in Hyderabad. This show is available in English, Hindi, Telugu, Tamil, Kannada, Malayalam and Marathi. Dholakpur is a fictional town in Chhota Bheem series. TBA. It is located in the fictional country of Australia, which was created by the government to promote the round world agenda. It is also called 'Dholak-poor' because Bheem spends their GDP on annual vacations to the Donald Trump Amusement Park. Jun 30, 2020 · Bheem and his friends are usually involved in protecting the village from various evil forces. The extremely strong Bheem lives in a fictional town called Dholakpur set in rural India. Apr 25, 2025 · The animated movie 'Chhota Bheem and the Curse of Damyaan' is set in the fictional city of Dholakpur. In the story, Dholakpur is a vibrant kingdom where the protagonist, Chhota Bheem, and his friends live.


In [9]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


### **Shell Tool**

In [10]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

results = shell_tool.invoke('ls')

print(results)

Executing command:
 ls
sample_data



/usr/local/lib/python3.13/dist-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


# **Custom Tool**

### **Using @Tool Decorator**

In [11]:
from langchain_core.tools import tool

In [12]:
def multiply(a,b):
  """multiply two numbers"""
  return a*b

In [13]:
def multiply(a:int,b:int)-> int:
  """multiply two numbers"""
  return a*b

In [14]:
@tool
def multiply(a:int,b:int)-> int:
  """multiply two numbers"""
  return a*b

In [15]:
result=multiply.invoke({"a":3,"b":6})

In [16]:
result

18

In [17]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [18]:
print(multiply.args_schema.model_json_schema())

{'description': 'multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


### **Using StructuredTool**

In [20]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [21]:
class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

/tmp/ipykernel_2541/3279971488.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  a: int = Field(required=True, description="The first number to add")
/tmp/ipykernel_2541/3279971488.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  b: int = Field(required=True, description="The second number to add")


In [22]:
def multiply_func(a: int, b: int) -> int:
    return a * b

In [23]:
multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

In [24]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


### **Using BaseTool Class**

In [25]:
from langchain_core.tools import BaseTool
from typing import Type

In [26]:
# arg schema using pydantic

class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

/tmp/ipykernel_2541/908171234.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  a: int = Field(required=True, description="The first number to add")
/tmp/ipykernel_2541/908171234.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  b: int = Field(required=True, description="The second number to add")


In [27]:
class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b

In [28]:
multiply_tool = MultiplyTool()

In [29]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


# **Tool Kit**

In [35]:
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

@tool
def divide(a: int, b: int) -> int:
    """Divide two numbers"""
    return a/b

In [36]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply,divide]

In [37]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)


add => Add two numbers
multiply => Multiply two numbers
divide => Divide two numbers
